In [ ]:
import pandas as pd

df_bruto = pd.read_csv("../data/vendas.csv", encoding="latin1")

# 1. Vizualização e Inspeção

display(df_bruto.head())
display(df_bruto.tail())
display(df_bruto.sample())
print(df_bruto.info())
print("Valores duplicados: ", df_bruto.duplicated().sum())

print(df_bruto["STATE"].unique()) # O resultado mostrou que a coluna não está organizada globalmente.
print(df_bruto["TERRITORY"].unique()) # o resultado mostrou que a coluna mistura categorias diferentes de território.
print(df_bruto["STATUS"].unique())
print(df_bruto["PRODUCTLINE"].unique())

# a função .unique() deve ser utilizada com critério: em colunas categóricas e que podem impactar em análises. O critério evita esforço desnecessário.



# A base de dados é internacional, portanto, podem haver diferenças de padronização. Um aprendizado importante com esse projeto: nulos e colunas fora do padrâo não necessariamente implicam erro, o mais importante é manter a base de dados funcional.

In [ ]:
# 2. Tratamento e Preparação dos Dados

df_entermediario = df_bruto.copy()


# Tipos de Dados
df_entermediario["ORDERDATE"] = pd.to_datetime(df_entermediario["ORDERDATE"])


# Padronização

colunas_texto = df_entermediario.select_dtypes(include="object").columns # Esse comando cria uma coleção com os nomes das colunas dtype object (tipo texto).

for coluna in colunas_texto:
    df_entermediario[coluna] = df_entermediario[coluna].str.strip()

df_entermediario["STATE"] = df_entermediario["STATE"].str.upper() # Primeiramente, deixei tudo maiúsculo, pois assim fica mais fácil de mapaer com .replace().

dicionario_padrao = {
    'NY': "Nova York",
    'CA': "California",
    'VICTORIA': "Victoria",
    'NJ': "Nova Jersey",
    'CT': "Connecticut",
    'MA': "Massachusetts",
    'PA': "Pennsylvania",
    'NSW': "New South Wales",
    'QUEENSLAND': "Queensland",
    'BC': "British Columbia",
    'TOKYO': "Tokyo",
    'NH': "New Hampshire",
    'QUEBEC': "Quebec",
    'OSAKA': "Osaka",
    'ISLE OF WIGHT': "Isle Of Wight",
    'NV': "Nevada",
}

df_entermediario["STATE"] = df_entermediario["STATE"].replace(dicionario_padrao) # Não utilizei .map() porque a .replace() deixa como está quando um valor específico não aparece no dicionário.

# Fiz essa padronização por questões de aprendizado. Porém, é preciso frisar que o ato de padronizar (ou não) depende das necessidades do projeto.


# Duplicados: conferi, no primeiro notebook, que não existem valores duplicados.


# Nulos: a partir da inspeção feita sobre df_row, percebi que os valores nulos existentes não necessariamente indicam erro. Essa determinação só pode ser feita a partir de informações do negócio.


display(df_entermediario)

display(df_entermediario.info())


# A coluna endereço não tem um padrão devido ao fato do dataset ser internacional. Novamente, o mais importante é que isso não interfira na capacidade de análise.

In [ ]:
# Validação de Consistência

# Vendas Inconsistentes
df_entermediario["SALES_CALCULADO"] = df_entermediario["QUANTITYORDERED"] * df_entermediario["PRICEEACH"]

inconsistentes = df_entermediario[
   (df_entermediario["SALES"] - df_entermediario["SALES_CALCULADO"]).abs() > 0.01
] # A função .abs() pega o valor absoluto. Incosistências só vão ser apontadas caso a diferença seja maior que 1 centavo.

print("Registros com inconsistência em SALES:", len(inconsistentes))

df_entermediario = df_entermediario.drop(columns=["SALES_CALCULADO"]) # Excluí SALES_CALCULADO para deixar o df_staging organizado.

# Preços Inválidos
print("PRICEEACH <= 0:", len(df_entermediario[df_entermediario["PRICEEACH"] <= 0]))

# Quantidade Inválida
print("QUANTITYORDERED <= 0:", len(df_entermediario[df_entermediario["QUANTITYORDERED"] <= 0]))

# Vendas Inválidas
print("SALES <= 0:", len(df_entermediario[df_entermediario["SALES"] <= 0]))

# Outras verificações (nulos, datas e categorias) foram realizadas durante a etapa de Data Profiling e não apresentaram inconsistências relevantes.

In [ ]:
# DataFrame para Análise

df_analise = df_entermediario.copy()

# A criação de datafremes a cada parte do processo é uma boa prática, pois conserva o dataset.

In [ ]:
# Análise - Receita por Produto

analise_produto = df_analise[["PRODUCTLINE", "SALES"]].groupby("PRODUCTLINE").sum()
analise_produto = analise_produto.sort_values(by="SALES", ascending=False)
analise_produto = analise_produto.reset_index() # após o groupby, é uma boa prática alterar a coluna indexada novamente para texto


display(analise_produto)

## Insight — Receita por Linha de Produto

A linha de produto **Classic Cars** concentra a maior parte da receita, apresentando desempenho significativamente superior às demais categorias.

Isso indica uma forte dependência do negócio nessa linha específica, o que pode representar um risco estratégico caso haja queda na demanda.

Além disso, a distribuição desigual da receita sugere baixa diversificação, com categorias como *Trains* e *Ships* tendo pouca participação no faturamento total.

In [ ]:
# Análise - Faturamento por País

analise_pais = df_analise[["COUNTRY", "SALES"]].groupby("COUNTRY").sum()
analise_pais = analise_pais.sort_values(by="SALES", ascending=False)
analise_pais = analise_pais.reset_index()

display(analise_pais)

### Insight — Receita por País

A análise de receita por país revela forte concentração nos Estados Unidos, que representam o principal mercado da empresa. Essa dependência indica um risco estratégico, uma vez que o desempenho do negócio está altamente ligado a esse país.

Embora haja presença internacional, não existe um segundo mercado dominante claro, com países como Espanha e França apresentando receitas relevantes, porém significativamente inferiores.

Além disso, observa-se uma longa cauda de países com baixo faturamento, o que pode indicar menor penetração nesses mercados ou oportunidades de expansão internacional.

In [ ]:
# Análise - Faturamento por Cliente
analise_cliente_faturamento = df_analise[["CUSTOMERNAME", "SALES"]].groupby("CUSTOMERNAME").sum()
analise_cliente_faturamento = analise_cliente_faturamento.sort_values(by="SALES", ascending=False)
analise_cliente_faturamento = analise_cliente_faturamento.reset_index()

display(analise_cliente_faturamento)

# O dataset não possui ID para cada cliente, então a melhor alternativa foi utilizar a coluna CUSTOMERNAME.

### Insight — Receita por Cliente

A análise de faturamento por cliente revela uma forte concentração de receita em poucos clientes, com destaque para *Euro Shopping Channel* e *Mini Gifts Distributors Ltd.*, que apresentam valores significativamente superiores aos demais.

Essa concentração indica uma dependência relevante de um pequeno grupo de clientes para a geração de receita, o que pode representar um risco estratégico para o negócio. A perda ou redução de compras desses clientes teria impacto direto no faturamento total.

Além disso, observa-se uma longa cauda de clientes com baixo volume de compras, sugerindo uma base ampla, porém com baixa contribuição individual. Isso pode indicar oportunidades de crescimento nesses clientes menores ou a necessidade de estratégias de segmentação e fidelização.

In [ ]:
# Análise - Faturamento por Mês

analise_mes = df_analise.groupby(df_staging["ORDERDATE"].dt.to_period("M"))["SALES"].sum() # cria uma Series
analise_mes = analise_mes.reset_index() # utilizar a função .reset_index() faz criar um df
analise_mes = analise_mes.sort_values(by="ORDERDATE", ascending=True)


display(analise_mes)
print(analise_mes.plot()) # A função .plot() cria uma gráfico sobre analise_mes.

### Insight — Receita ao Longo do Tempo

A análise da receita ao longo do tempo revela um padrão claro de sazonalidade, com picos significativos de faturamento nos meses de outubro e, principalmente, novembro, em todos os anos analisados.

O mês de novembro apresenta consistentemente o maior volume de vendas, indicando forte influência de fatores sazonais, como datas comerciais específicas ou aumento da demanda no final do ano.

Após esses picos, observa-se uma queda acentuada em dezembro, sugerindo possível antecipação de compras ou redução da atividade após o período de alta.

Além disso, ao longo dos anos, há indícios de crescimento no faturamento total, embora com variações mensais, o que reforça a importância de estratégias comerciais alinhadas ao calendário para maximizar resultados nos períodos de maior demanda.

In [ ]:
# Análise - Distribuição de Status

analise_status = df_analise[["STATUS", "ORDERNUMBER"]].groupby("STATUS").count()
analise_status = analise_status.sort_values(by="ORDERNUMBER", ascending=False)
analise_status = analise_status.reset_index()


display(analise_status)

### Insight — Distribuição de Status dos Pedidos

A análise da distribuição de status dos pedidos mostra que a grande maioria está como *Shipped*, indicando uma operação eficiente, com alto volume de pedidos concluídos com sucesso.

Os pedidos cancelados representam uma pequena parcela do total, o que sugere baixa taxa de cancelamento e bom desempenho no processo de vendas e entrega.

Além disso, a presença de pedidos com status como *On Hold*, *In Process* e *Disputed* indica que há uma pequena fração da operação que ainda está em andamento ou enfrenta algum tipo de problema, o que pode representar oportunidades de melhoria nos processos internos.

De forma geral, os dados indicam uma operação saudável, com predominância de pedidos finalizados e baixa incidência de falhas críticas.